# Chapter 5: Updates and Transactions

**Level 2: Modeling & Querying** · ⏱ about 2 hours · Dataset: *Recipes & Nutrition* (+ generated ratings)

### What you will learn

1. The ways to change data: `add`/`remove`, **SPARQL Update**, and **graph operations**
2. **INSERT DATA / DELETE DATA** for fixed triples
3. **DELETE / INSERT … WHERE** for pattern-based changes, with parameters
4. Managing whole graphs: **CLEAR, DROP, COPY, MOVE, ADD**
5. **Transaction semantics**: atomicity, visibility, commit counts, rollback, and **concurrent writers**
6. **Hands-on 1:** an **idempotent upsert** for recipes
7. **Hands-on 2:** **batch-loading** 10,000 ratings, and why batching matters (hundreds of times faster)

In [1]:
import sys
import time
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import stardog
from rdflib import URIRef

from kg import client, sparql

conn = client.connect()
PREFIX = 'urn:tutorial:ch05'
GRAPH = f'{PREFIX}:recipes'
DATA = Path('..', 'data', 'recipes')

with client.transaction(conn) as stats:
    conn.clear(graph_uri=GRAPH)
    conn.add(stardog.content.File(str(DATA / 'recipes.ttl')), graph_uri=GRAPH)
print('Loaded recipes:', stats)

PREFIXES = '''PREFIX rec: <http://example.org/recipes#>
PREFIX recipe: <http://example.org/recipes/recipe/>
PREFIX ingredient: <http://example.org/recipes/ingredient/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
'''
def q(query, graphs=GRAPH, **params):
    return sparql.run_query(conn, PREFIXES + query, graphs=graphs, **params)

def triples(graph=GRAPH):
    return int(q('SELECT (COUNT(*) AS ?n) { ?s ?p ?o }', graphs=graph).n[0])

Loaded recipes: {'added': 1020, 'removed': 0}


`client.transaction(conn)` now yields a small **stats** dictionary. After the block commits, it holds Stardog's commit result: how many triples were **added** and **removed**. We'll use it throughout the chapter to see exactly what each change did.

## 5.1 Three ways to change data

| Way | pystardog | Good for |
|---|---|---|
| **Content** | `conn.add(File/Raw, graph_uri=…)`, `conn.remove(…)` | Loading files and bulk data (Chapter 3) |
| **SPARQL Update** | `conn.update('INSERT … / DELETE …')` | Targeted and pattern-based changes |
| **Graph operations** | `conn.clear(graph_uri)`, or `CLEAR / DROP / COPY / MOVE / ADD` via `update()` | Whole graphs: reset, back up, promote |

## 5.2 INSERT DATA and DELETE DATA

The `DATA` forms add or remove **fixed** triples: no variables, no patterns. Always wrap them in `GRAPH <…> { }`, otherwise they go to the default graph.

In [2]:
conn.update(PREFIXES + f'''
INSERT DATA {{
  GRAPH <{GRAPH}> {{
    recipe:masala-omelette a rec:Recipe ;
        rec:name "Masala Omelette"@en , "Omelette masala"@fr ;
        rec:cuisine "Indian" ; rec:prepMinutes 15 ; rec:servings 1 ;
        rec:hasIngredient ingredient:egg , ingredient:onion .
  }}
}}''')
q('SELECT ?p ?o { recipe:masala-omelette ?p ?o } ORDER BY ?p')

,p,o
0,http://example.org/recipes#cuisine,Indian
1,http://example.org/recipes#hasIngredient,http://example.org/recipes/ingredient/egg
2,http://example.org/recipes#hasIngredient,http://example.org/recipes/ingredient/onion
3,http://example.org/recipes#name,Omelette masala
4,http://example.org/recipes#name,Masala Omelette
5,http://example.org/recipes#prepMinutes,15
6,http://example.org/recipes#servings,1
7,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://example.org/recipes#Recipe


Outside a transaction, each `update()` is **committed immediately** (auto-commit).

`DELETE DATA` removes exact triples. Say the servings were wrong:

In [3]:
conn.update(PREFIXES + f'''
DELETE DATA {{ GRAPH <{GRAPH}> {{ recipe:masala-omelette rec:servings 1 . }} }} ;
INSERT DATA {{ GRAPH <{GRAPH}> {{ recipe:masala-omelette rec:servings 2 . }} }}''')
q('SELECT ?servings { recipe:masala-omelette rec:servings ?servings }')

,servings
0,2


Several operations separated by `;` run as **one request**.

`DELETE DATA` needs the **exact value**, including its datatype: `rec:servings "1"` (a string) would not match `rec:servings 1` (an integer), and nothing would be deleted, **with no error**. When you don't know the current value, use a pattern instead:

## 5.3 DELETE / INSERT … WHERE

The general update has three parts:

```sparql
DELETE { template }      ← triples to remove, for each match
INSERT { template }      ← triples to add, for each match
WHERE  { pattern }       ← find the matches first
```

The `WHERE` runs **first**, then both templates are applied to every match. Either template may be left out.

### Change a value you don't know

In [4]:
conn.update(PREFIXES + f'''
DELETE {{ GRAPH <{GRAPH}> {{ recipe:masala-omelette rec:prepMinutes ?old }} }}
INSERT {{ GRAPH <{GRAPH}> {{ recipe:masala-omelette rec:prepMinutes 12 }} }}
WHERE  {{ GRAPH <{GRAPH}> {{ recipe:masala-omelette rec:prepMinutes ?old }} }}''')
q('SELECT ?minutes { recipe:masala-omelette rec:prepMinutes ?minutes }')

,minutes
0,12


### Change many things at once

Tag every recipe ready in 20 minutes or less as `quick`, and double the servings of all Italian recipes (a party menu). `WITH <graph>` sets the graph for the whole update, which is shorter than repeating `GRAPH` blocks:

In [5]:
with client.transaction(conn) as stats:
    conn.update(PREFIXES + f'''
    WITH <{GRAPH}>
    INSERT {{ ?r rec:tag "quick" }}
    WHERE  {{ ?r a rec:Recipe ; rec:prepMinutes ?m  FILTER(?m <= 20) }}''')

    conn.update(PREFIXES + f'''
    WITH <{GRAPH}>
    DELETE {{ ?r rec:servings ?s }}
    INSERT {{ ?r rec:servings ?doubled }}
    WHERE  {{ ?r rec:cuisine "Italian" ; rec:servings ?s  BIND(?s * 2 AS ?doubled) }}''')
print('Commit:', stats)

q('''SELECT ?name ?servings ?tag {
  ?r rec:name ?name ; rec:servings ?servings  FILTER(lang(?name) = "en")
  OPTIONAL { ?r rec:tag ?tag } }
ORDER BY ?name''')

Commit: {'added': 9, 'removed': 3}


,name,servings,tag
0,Chicken Stir Fry,3,NaN
1,Chocolate Brownies,12,NaN
2,Crispy Chicken Tenders,3,NaN
3,Grilled Salmon,2,quick
4,Lupin Flatbread,4,quick
5,Margherita Pizza,4,NaN
6,Masala Omelette,2,quick
7,Moules Marinieres,2,NaN
8,Pad Thai,2,NaN
9,Pancakes,4,quick


6 `quick` tags were added, and 3 Italian servings replaced (3 removed, 3 added): 9 added and 3 removed in total.

### Parameters in updates

Updates accept **bindings** just like queries, so values never need to be pasted into the text. Rename a cuisine:

In [6]:
RENAME_CUISINE = PREFIXES + f'''
WITH <{GRAPH}>
DELETE {{ ?r rec:cuisine ?old }}
INSERT {{ ?r rec:cuisine ?new }}
WHERE  {{ ?r rec:cuisine ?old }}'''

conn.update(RENAME_CUISINE, bindings={'old': sparql.term('Mediterranean'), 'new': sparql.term('Greek')})
q('SELECT ?cuisine (COUNT(?r) AS ?recipes) { ?r rec:cuisine ?cuisine } GROUP BY ?cuisine ORDER BY ?cuisine')

,cuisine,recipes
0,American,3
1,Chinese,1
2,French,2
3,German,1
4,Greek,2
5,Indian,1
6,Italian,3
7,Thai,2


### Save derived facts

In Chapter 4, `CONSTRUCT` derived `recipe → allergen` links without storing them. `INSERT … WHERE` stores the same thing (**materialization**):

In [7]:
with client.transaction(conn) as stats:
    conn.update(PREFIXES + f'''
    WITH <{GRAPH}>
    INSERT {{ ?r rec:containsAllergen ?a }}
    WHERE  {{ ?r a rec:Recipe ; rec:hasIngredient/rec:hasAllergen ?a }}''')
print('Materialized:', stats)

Materialized: {'added': 40, 'removed': 0}


Materialized facts are **fast to query** but go **stale** when the ingredients change. They have to be recomputed. Chapter 8's reasoner derives them at query time instead, so they're always current. That trade-off comes up again in Chapter 15.

### DELETE WHERE: the shorthand

When the delete template equals the pattern, write it once:

In [8]:
conn.update(PREFIXES + f'DELETE WHERE {{ GRAPH <{GRAPH}> {{ ?r rec:tag ?tag }} }}')
print('Tags left:', int(q('SELECT (COUNT(*) AS ?n) { ?r rec:tag ?t }').n[0]))

Tags left: 0


> 🔁 **Neo4j equivalent:** `CREATE` ≈ `INSERT DATA`; `MATCH … SET r.servings = r.servings * 2` ≈ `DELETE/INSERT … WHERE`; `MATCH … DETACH DELETE` ≈ `DELETE WHERE`. Cypher's `SET` replaces a property value in place; in RDF you delete the old triple and insert the new one.

## 5.4 Managing whole graphs

| Operation | Effect |
|---|---|
| `CLEAR GRAPH <g>` | Remove all triples in `g` (pystardog: `conn.clear(graph_uri=g)`) |
| `DROP GRAPH <g>` | Remove the graph (same as CLEAR in Stardog) |
| `COPY <a> TO <b>` | Make `b` an exact copy of `a` (replaces `b`) |
| `ADD <a> TO <b>` | Add `a`'s triples to `b` (keeps `b`'s) |
| `MOVE <a> TO <b>` | Copy, then drop `a` |

`SILENT` (e.g. `DROP SILENT GRAPH`) ignores "graph doesn't exist" errors.

A classic use is a **backup before a risky change**:

In [9]:
BACKUP = f'{PREFIX}:recipes:backup'
conn.update(f'COPY <{GRAPH}> TO <{BACKUP}>')
print('backup:', triples(BACKUP), 'triples')

conn.update(PREFIXES + f'DELETE WHERE {{ GRAPH <{GRAPH}> {{ ?r rec:cuisine "Italian" ; ?p ?o }} }}')  # oops
print('after the mistake:', triples(), 'triples')

conn.update(f'COPY <{BACKUP}> TO <{GRAPH}>')                                                      # restore
conn.update(f'DROP SILENT GRAPH <{BACKUP}>')
print('restored:', triples(), 'triples')

backup: 1065 triples


after the mistake: 994 triples


restored: 1065 triples


## 5.5 Transactions

A transaction groups changes so they **all happen or none do**.

```python
conn.begin()        # start
conn.update(...)    # any number of changes
conn.add(...)
conn.commit()       # make them permanent, or conn.rollback() to discard them
```

### Visibility: who sees what, when

Inside a transaction, **your** connection sees its own uncommitted changes. **Other** connections keep seeing the old data until you commit:

In [10]:
def recipe_count(c):
    return int(sparql.run_query(c, 'PREFIX rec: <http://example.org/recipes#> SELECT (COUNT(?r) AS ?n) { ?r a rec:Recipe }',
                                graphs=GRAPH).n[0])

other = client.connect()                     # a second user
conn.begin()
conn.update(PREFIXES + f'INSERT DATA {{ GRAPH <{GRAPH}> {{ recipe:toast a rec:Recipe . }} }}')
print(f'during the transaction: mine sees {recipe_count(conn)}, other sees {recipe_count(other)}')
print('commit:', conn.commit())
print(f'after commit:           mine sees {recipe_count(conn)}, other sees {recipe_count(other)}')

during the transaction: mine sees 16, other sees 15


commit: {'added': 1, 'removed': 0}


after commit:           mine sees 16, other sees 16


`commit()` returns what changed. That's handy for logging and tests.

### Rollback

In [11]:
before = triples()
conn.begin()
conn.update(PREFIXES + f'DELETE WHERE {{ GRAPH <{GRAPH}> {{ ?s ?p ?o }} }}')    # delete EVERYTHING in the graph
print('inside the transaction:', triples(), 'triples')
conn.rollback()
print(f'after rollback: {triples()} triples (was {before})')

inside the transaction: 0 triples


after rollback: 1066 triples (was 1066)


### Failures roll back automatically

As in Chapter 3: if any request in a transaction fails, Stardog aborts the whole transaction. The earlier changes in it are discarded too. `client.transaction()` keeps the connection usable afterwards:

In [12]:
before = triples()
try:
    with client.transaction(conn):
        conn.update(PREFIXES + f'INSERT DATA {{ GRAPH <{GRAPH}> {{ recipe:toast rec:prepMinutes 5 . }} }}')   # fine
        conn.update(PREFIXES + f'INSERT DATA {{ GRAPH <{GRAPH}> {{ recipe:toast rec:prepMinutes }} }}')     # syntax error
except stardog.exceptions.StardogException as e:
    print('failed:', e.http_code, str(e)[:90])
print(f'triples: {triples()} (was {before}), so the first INSERT was undone too')

failed: 400 [400] QEIVR2: com.complexible.stardog.plan.eval.ExecutionException: Data block for update 


triples: 1066 (was 1066), so the first INSERT was undone too


### Concurrent writers

Stardog runs transactions with **snapshot isolation**: each transaction works on a snapshot of the data taken when it began. When two transactions change the same data, the database option `transaction.write.conflict.strategy` decides what happens:

In [13]:
with client.admin() as admin:
    admin.database(client.load_settings().database).get_options(
        'transaction.isolation', 'transaction.write.conflict.strategy')

**`LAST_COMMIT_WINS`** means both commits succeed, and the later one overwrites the earlier one. Watch what that does to a view counter that two users increment at the same time:

In [14]:
COUNTER = f'{PREFIX}:counter'
INCREMENT = PREFIXES + f'''
WITH <{COUNTER}>
DELETE {{ recipe:pad-thai rec:views ?v }}
INSERT {{ recipe:pad-thai rec:views ?next }}
WHERE  {{ recipe:pad-thai rec:views ?v  BIND(?v + 1 AS ?next) }}'''

def reset_counter():
    conn.update(f'DROP SILENT GRAPH <{COUNTER}>')
    conn.update(PREFIXES + f'INSERT DATA {{ GRAPH <{COUNTER}> {{ recipe:pad-thai rec:views 10 }} }}')

def views():
    return int(q('SELECT ?v { recipe:pad-thai rec:views ?v }', graphs=COUNTER).v[0])

# Two overlapping transactions
reset_counter()
conn.begin(); other.begin()
conn.update(INCREMENT); other.update(INCREMENT)
conn.commit(); other.commit()
print('overlapping transactions: 10 + 1 + 1 =', views())

# Two short auto-commit updates, one after the other
reset_counter()
conn.update(INCREMENT); other.update(INCREMENT)
print('separate auto-commits:    10 + 1 + 1 =', views())

overlapping transactions: 10 + 1 + 1 = 11


separate auto-commits:    10 + 1 + 1 = 12


With overlapping transactions, both saw `10` in their snapshot and both wrote `11`: **one increment was lost**, and no error was raised. As separate short updates, each saw the other's result.

**Guidelines**

- Keep write transactions **short**. Don't hold one open while a user thinks or a slow API call runs.
- Do read-modify-write **in one update** on the server (`BIND(?v + 1 …)`), not by reading in Python and writing back.
- If lost updates are unacceptable, an admin can set `transaction.write.conflict.strategy` to **`ABORT_ON_CONFLICT`**. The second commit then fails, and you retry it. It's a database setting, so we don't change it here.

> 🔁 **Neo4j equivalent:** Neo4j takes **write locks**, so the second writer waits and then sees the first one's change. Stardog doesn't block; it resolves conflicts at commit time according to the strategy above.

## 5.6 Hands-on 1: an idempotent upsert

Imagine recipes arrive from a CMS as JSON, and the same recipe may be sent many times, sometimes changed. We need an **upsert** that:

- creates the recipe if it's new,
- **replaces** its description if it exists, without leaving old values or duplicating the blank-node ingredient lines (Chapter 3's trap),
- is **idempotent**: sending the same recipe twice leaves the graph unchanged,
- is **atomic**: a failure leaves the old version intact.

The recipe is:

**delete everything about the resource, including its blank nodes, then add the new description, in one transaction.**

Two small modules implement it:

- `kg/recipes.py`: the IRI conventions from Chapter 3, and `recipe_to_graph(dict)`
- `kg/updates.py`: `delete_resource()` and `replace_resource()`

In [15]:
from kg import recipes, updates

print(Path(updates.__file__).read_text())

"""Idempotent writes: replace or delete everything a graph says about one resource."""
import stardog
from rdflib import Graph, URIRef

from kg import client
from kg.sparql import term

# Everything about ?s in the graph, plus the triples of blank nodes it points to
# (e.g. ingredient lines), so they don't linger or pile up as duplicates.
_DELETE_RESOURCE = '''
DELETE {{ GRAPH <{graph}> {{ ?s ?p ?o . ?o ?p2 ?o2 . }} }}
WHERE  {{ GRAPH <{graph}> {{ ?s ?p ?o  OPTIONAL {{ ?o ?p2 ?o2  FILTER(isBlank(?o)) }} }} }}
'''


def delete_resource(conn: stardog.Connection, graph: str, subject: URIRef) -> None:
  """Delete the resource's triples (and its blank nodes). Call inside a transaction."""
  conn.update(_DELETE_RESOURCE.format(graph=graph), bindings={'s': term(subject)})


def replace_resource(conn: stardog.Connection, graph: str, subject: URIRef, description: Graph) -> dict:
  """Upsert: replace what `graph` says about `subject` with `description`, atomically.

  Running it again with the s

The `OPTIONAL { ?o ?p2 ?o2 FILTER(isBlank(?o)) }` part also deletes the triples **inside** blank nodes the resource points to: the ingredient lines. The subject is passed as a **binding** (`?s`), not pasted into the text.

Let's upsert a recipe that arrives from the "CMS":

In [16]:
omelette = {
    'id': 'R15', 'name': 'Masala Omelette', 'name_fr': 'Omelette masala',
    'cuisine': 'Indian', 'prep_minutes': 15, 'servings': 1, 'created': '2025-12-01',
    'description': 'Spiced omelette with onion.',
    'ingredients': [('egg', 3, 'piece'), ('onion', 1, 'piece')],
}
IRI = recipes.recipe_iri(omelette)

def show():
    return q('''SELECT ?servings ?minutes (COUNT(?line) AS ?lines) {
      ?r rec:servings ?servings ; rec:prepMinutes ?minutes ; rec:hasIngredientLine ?line }
      GROUP BY ?servings ?minutes''', r=IRI)

print('1st upsert:', updates.replace_resource(conn, GRAPH, IRI, recipes.recipe_to_graph(omelette)), '| triples:', triples())
display(show())

1st upsert: {'added': 21, 'removed': 9} | triples: 1078


,servings,minutes,lines
0,1,15,2


Earlier (5.2) we inserted a masala omelette **by hand** with servings 2 and prep time 12. The upsert **replaced** that description completely, so there's one value of each and no leftovers.

Now send the **same** recipe again, and then a **changed** one:

In [17]:
print('2nd upsert (same):   ', updates.replace_resource(conn, GRAPH, IRI, recipes.recipe_to_graph(omelette)), '| triples:', triples())

omelette['prep_minutes'] = 10
omelette['ingredients'].append(('garlic', 1, 'clove'))
print('3rd upsert (changed):', updates.replace_resource(conn, GRAPH, IRI, recipes.recipe_to_graph(omelette)), '| triples:', triples())
show()

2nd upsert (same):    {'added': 21, 'removed': 21} | triples: 1078


3rd upsert (changed): {'added': 27, 'removed': 21} | triples: 1084


,servings,minutes,lines
0,1,10,3


- **Same recipe again:** the triple count is unchanged. The commit reports equal numbers removed and added, because the blank-node lines are always recreated with new IDs, but the **content** is identical.
- **Changed recipe:** the new prep time and a third ingredient line, with no duplicates.

Deleting uses the same helper:

In [18]:
with client.transaction(conn) as stats:
    updates.delete_resource(conn, GRAPH, IRI)
print('deleted:', stats, '| recipe still there?', sparql.ask(conn, 'ASK { ?r ?p ?o }', graphs=GRAPH, r=IRI))

deleted: {'added': 0, 'removed': 27} | recipe still there? False


> 🔁 **Neo4j equivalent:** `MERGE (r:Recipe {id: 'R15'}) SET r += $props`. `MERGE` creates or matches on a key. RDF has no built-in upsert, so this delete-and-add-in-one-transaction pattern is how it's done.

## 5.7 Hands-on 2: batch-loading 10,000 ratings

Next we add **user ratings** to the recipes. Chapter 6 uses them for aggregates, and Chapter 15 for performance. We generate 1,000 users and 10,000 ratings (1–5 stars). Each rating is its own resource:

```turtle
rating:u0042-pad-thai a rec:Rating ;
    rec:byUser user:u0042 ; rec:ofRecipe recipe:pad-thai ;
    rec:score 4 ; rec:ratedOn "2025-06-14"^^xsd:date .
```

In [19]:
import random
from datetime import date, timedelta

from rdflib import Literal
from rdflib.namespace import RDF, XSD

random.seed(42)
recipe_iris = q('SELECT ?r { ?r a rec:Recipe ; rec:recipeId ?id }').r.tolist()
popularity = {r: random.uniform(2.3, 4.6) for r in recipe_iris}   # each recipe has its own typical score

pairs = random.sample([(u, r) for u in range(1, 1001) for r in recipe_iris], 10_000)
ratings = recipes.new_graph()
for u in range(1, 1001):
    ratings.add((recipes.USER[f'u{u:04d}'], RDF.type, recipes.REC.User))
for u, r in pairs:
    rating = recipes.RATING[f'u{u:04d}-{r.rsplit("/", 1)[1]}']
    score = min(5, max(1, round(random.gauss(popularity[r], 0.9))))
    day = date(2025, 1, 1) + timedelta(days=random.randrange(365))
    for p, o in [(RDF.type, recipes.REC.Rating), (recipes.REC.byUser, recipes.USER[f'u{u:04d}']),
                 (recipes.REC.ofRecipe, URIRef(r)), (recipes.REC.score, Literal(score)),
                 (recipes.REC.ratedOn, Literal(day, datatype=XSD.date))]:
        ratings.add((rating, p, o))

rating_lines = ratings.serialize(format='nt').splitlines()   # one triple per line: easy to batch
print(f'{len(ratings):,} triples ({len(pairs):,} ratings, 1,000 users)')

51,000 triples (10,000 ratings, 1,000 users)


### Strategy 1: one INSERT per rating

The obvious approach: loop and insert each rating. We time 30 of them and extrapolate:

In [20]:
RATINGS = f'{PREFIX}:ratings'
conn.update(f'DROP SILENT GRAPH <{RATINGS}>')

by_rating = {}
for line in rating_lines:
    if '/rating/' in line.split(' ', 1)[0]:
        by_rating.setdefault(line.split(' ', 1)[0], []).append(line)
sample = list(by_rating.values())[:30]

start = time.perf_counter()
for rating_triples in sample:
    conn.update(f'INSERT DATA {{ GRAPH <{RATINGS}> {{ {" ".join(rating_triples)} }} }}')
per_rating = (time.perf_counter() - start) / len(sample)
one_by_one = per_rating * 10_000
print(f'{per_rating * 1000:.0f} ms per rating  ->  10,000 ratings ≈ {one_by_one / 60:.0f} minutes')

304 ms per rating  ->  10,000 ratings ≈ 51 minutes


Every `update()` is a separate HTTP request **and** a separate commit. The cost is almost all overhead, not data.

### Strategy 2: one bulk load

Send everything as one N-Triples document in one transaction:

In [21]:
conn.update(f'DROP SILENT GRAPH <{RATINGS}>')
start = time.perf_counter()
with client.transaction(conn) as stats:
    conn.add(stardog.content.Raw('\n'.join(rating_lines).encode(), 'application/n-triples'), graph_uri=RATINGS)
bulk = time.perf_counter() - start
print(f'{bulk:.1f} s for {stats["added"]:,} triples')

6.4 s for 51,000 triples


### Strategy 3: chunks

One huge transaction has drawbacks for really big loads: it's all-or-nothing (a failure at 99% loses everything), it holds a lot of memory, and you get no progress. **Chunks** of a few thousand triples, each in its own transaction, are the usual compromise:

In [22]:
def load_in_chunks(conn, lines, graph, chunk_size=10_000):
    '''Load N-Triples lines in chunks, one transaction per chunk. Returns triples added.'''
    added = 0
    for i in range(0, len(lines), chunk_size):
        with client.transaction(conn) as stats:
            conn.add(stardog.content.Raw('\n'.join(lines[i:i + chunk_size]).encode(), 'application/n-triples'),
                     graph_uri=graph)
        added += stats['added']
        print(f'  chunk {i // chunk_size + 1}: {added:,} / {len(lines):,}')
    return added

conn.update(f'DROP SILENT GRAPH <{RATINGS}>')
start = time.perf_counter()
load_in_chunks(conn, rating_lines, RATINGS)
chunked = time.perf_counter() - start
print(f'{chunked:.1f} s')

  chunk 1: 10,000 / 51,000


  chunk 2: 20,000 / 51,000


  chunk 3: 30,000 / 51,000


  chunk 4: 40,000 / 51,000


  chunk 5: 50,000 / 51,000


  chunk 6: 51,000 / 51,000
6.7 s


N-Triples is ideal for chunking: **one triple per line**, so any group of lines is valid on its own. With blank nodes that wouldn't be safe, since one node's triples could end up in different chunks and become different nodes. The ratings use IRIs.

In [23]:
comparison = pd.DataFrame({
    'strategy': ['1. one INSERT per rating (extrapolated)', '2. one bulk load', '3. chunks of 10,000 triples'],
    'seconds': [one_by_one, bulk, chunked],
})
comparison['times slower than bulk'] = (comparison.seconds / bulk).round(1)
comparison.round(1)

,strategy,seconds,times slower than bulk
0,1. one INSERT per rating (extrapolated),3036.6,477.9
1,2. one bulk load,6.4,1.0
2,"3. chunks of 10,000 triples",6.7,1.1


**Takeaways**

- **Never** load data one statement at a time. The round trips and commits dominate.
- Use **one bulk `add`** for up to a few hundred thousand triples, and **chunks** beyond that, or when you need progress and restartability.
- For millions of triples, compress the file (`.nt.gz`) and load it with `File`, or use Stardog's server-side bulk loader when creating a database.

### Save the ratings for later chapters

Chapters 6 and 15 load the ratings from a compressed file:

In [24]:
import gzip

RATINGS_FILE = DATA / 'ratings.nt.gz'
with gzip.open(RATINGS_FILE, 'wt', encoding='utf-8') as f:
    f.write('\n'.join(rating_lines) + '\n')
print(f'Wrote {RATINGS_FILE} ({RATINGS_FILE.stat().st_size / 1024:.0f} KB, from {len(chr(10).join(rating_lines).encode()) / 1024:.0f} KB)')

Wrote ..\data\recipes\ratings.nt.gz (310 KB, from 7195 KB)


A quick look at the result: the best-rated recipes. `GROUP BY` and `AVG` are Chapter 6's topic.

In [25]:
q('''SELECT ?recipe (ROUND(AVG(?score) * 100) / 100 AS ?avg_score) (COUNT(?rating) AS ?ratings) {
  ?rating rec:ofRecipe ?r ; rec:score ?score .
  ?r rec:name ?recipe  FILTER(lang(?recipe) = "en") }
GROUP BY ?recipe ORDER BY DESC(?avg_score) LIMIT 5''', graphs=[GRAPH, RATINGS])

,recipe,avg_score,ratings
0,Margherita Pizza,4.23,736
1,Grilled Salmon,3.93,714
2,Lupin Flatbread,3.83,727
3,Pesto Pasta,3.74,714
4,Spaghetti Carbonara,3.44,696


Note `graphs=[GRAPH, RATINGS]`: the recipe names and the ratings live in **different named graphs**, and the query sees both merged (Chapter 3).

## 5.8 Exercises

**Exercise 1.** Every ingredient line with unit `"tsp"` should say `"teaspoon"`. Write one `DELETE/INSERT … WHERE` update, and check it with a count before and after.

<details><summary>Solution</summary>

```python
count = 'SELECT (COUNT(?l) AS ?n) { ?l rec:unit ?u } '
print(q(count + 'VALUES ?u { "tsp" }').n[0], q(count + 'VALUES ?u { "teaspoon" }').n[0])
conn.update(PREFIXES + f'''WITH <{GRAPH}>
DELETE {{ ?l rec:unit "tsp" }} INSERT {{ ?l rec:unit "teaspoon" }} WHERE {{ ?l rec:unit "tsp" }}''')
print(q(count + 'VALUES ?u { "tsp" }').n[0], q(count + 'VALUES ?u { "teaspoon" }').n[0])
```
</details>

**Exercise 2.** In a transaction, delete all **Thai** recipes (every triple about them), check the recipe count inside the transaction, then **roll back** and check again.

<details><summary>Solution</summary>

```python
conn.begin()
conn.update(PREFIXES + f'DELETE WHERE {{ GRAPH <{GRAPH}> {{ ?r rec:cuisine "Thai" ; ?p ?o }} }}')
print('inside:', recipe_count(conn))
conn.rollback()
print('after rollback:', recipe_count(conn))
```
</details>

**Exercise 3.** Why does `replace_resource()` delete and add in **one** transaction? What could a user see if they were two separate auto-commit steps?

<details><summary>Solution</summary>

Between the two commits the recipe would **not exist at all**. A query running at that moment would miss it, and if the add then failed, the recipe would be lost. In one transaction, other users see either the old version or the new one, never a gap.
</details>

**Exercise 4.** Use `load_in_chunks` with `chunk_size=2_500` to reload the ratings into a fresh graph `f'{PREFIX}:ratings2'`. How many chunks run, and does the triple count match?

<details><summary>Solution</summary>

```python
conn.update(f'DROP SILENT GRAPH <{PREFIX}:ratings2>')
load_in_chunks(conn, rating_lines, f'{PREFIX}:ratings2', chunk_size=2_500)   # 21 chunks
print(triples(f'{PREFIX}:ratings2') == triples(RATINGS))
```
</details>

In [26]:
# your answers here

## 5.9 Clean up

In [27]:
CLEAN_UP = True

graphs = q(f'SELECT DISTINCT ?g {{ GRAPH ?g {{ ?s ?p ?o }} FILTER(STRSTARTS(STR(?g), "{PREFIX}")) }}', graphs=None).g.tolist()
if CLEAN_UP:
    with client.transaction(conn):
        for g in graphs:
            conn.clear(graph_uri=g)
    print('Removed:', *graphs, sep='\n  ')
other.close()
conn.close()

Removed:
  urn:tutorial:ch05:recipes
  urn:tutorial:ch05:counter
  urn:tutorial:ch05:ratings


## Summary

- **INSERT DATA / DELETE DATA** change fixed triples. `DELETE DATA` silently does nothing if the value or datatype doesn't match exactly.
- **DELETE / INSERT … WHERE** changes data by pattern: the `WHERE` runs first, then both templates apply to every match. `WITH <g>` sets the graph; `DELETE WHERE` is the shorthand. Updates accept **bindings**.
- **Graph operations** (CLEAR, DROP, COPY, MOVE, ADD) manage whole graphs, e.g. a backup before a risky change.
- **Transactions** are atomic, and other connections see changes only after `commit()`. `commit()` reports added and removed counts, `rollback()` discards, and a failure discards the whole transaction.
- Stardog uses **snapshot isolation** and, on this database, **LAST_COMMIT_WINS**: overlapping transactions can **lose updates**. Keep transactions short and do read-modify-write in a single server-side update.
- **Upsert** = delete the resource's description, including its blank nodes, and add the new one, in one transaction (`updates.replace_resource`).
- **Batch** your loads: one bulk `add` was hundreds of times faster than one INSERT per rating. Use N-Triples chunks for very large or restartable loads.

**Next: Chapter 6, Advanced SPARQL.** Aggregates, subqueries, `BIND`, `VALUES`, property paths through the ingredient category hierarchy, and safe parameterized queries.